# Bitcoin Market Sentiment vs. Trader Performance

**Objective:** Explore the relationship between Bitcoin market sentiment (Fear & Greed Index) and trader performance on Hyperliquid, uncover hidden patterns, and derive insights that can inform sentiment-aware trading strategies.

**Datasets**
1. Bitcoin Market Sentiment Dataset — `Date`, `Classification` (Fear/Greed)
2. Historical Trader Data from Hyperliquid — `account`, `symbol`, `execution price`, `size`, `side`, `time`, `start position`, `event`, `closedPnL`, `leverage`, etc.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-darkgrid')


## 1. Load Data

In [1]:
trades = pd.read_csv('data/historical_data.csv')
sentiment = pd.read_csv('data/fear_greed_index.csv')

print("Trades:", trades.shape)
print("Sentiment:", sentiment.shape)
trades.head()


Trades: (211224, 16)
Sentiment: (2644, 4)


In [ ]:
sentiment.head()


## 2. Data Cleaning

- Convert `Closed PnL` and `Size USD` to numeric (some rows are non-numeric/blank).
- Parse `Timestamp IST` into a proper datetime and extract the trade date.
- Parse the sentiment `date` column and align it to the same daily granularity.


In [1]:
trades['Closed PnL'] = pd.to_numeric(trades['Closed PnL'], errors='coerce')
trades['Size USD'] = pd.to_numeric(trades['Size USD'], errors='coerce')
trades['date'] = pd.to_datetime(trades['Timestamp IST'], format='%d-%m-%Y %H:%M', errors='coerce')
trades['date_only'] = trades['date'].dt.date

sentiment['date_only'] = pd.to_datetime(sentiment['date']).dt.date

print("Missing Closed PnL:", trades['Closed PnL'].isna().sum())
print("Missing trade dates:", trades['date'].isna().sum())
print("Sentiment date range:", sentiment['date_only'].min(), "to", sentiment['date_only'].max())


Missing Closed PnL: 0
Missing trade dates: 0
Sentiment date range: 2018-02-01 to 2025-05-02


## 3. Aggregate Trades to Daily Level

Each trader/account can have many trades per day, so we roll trades up to one row per day before joining to the daily sentiment reading.


In [ ]:
daily = trades.groupby('date_only').agg(
    total_pnl=('Closed PnL', 'sum'),
    avg_trade_pnl=('Closed PnL', 'mean'),
    trades_count=('Closed PnL', 'count'),
    win_rate=('Closed PnL', lambda x: (x > 0).mean() * 100),
    avg_size_usd=('Size USD', 'mean')
).reset_index()

daily.head()


## 4. Merge Trades with Sentiment

In [1]:
merged = pd.merge(
    daily,
    sentiment[['date_only', 'value', 'classification']],
    on='date_only',
    how='inner'
)

print("Merged shape:", merged.shape)
merged.head()


Merged shape: (479, 8)


## 5. Performance by Sentiment Classification

Group the merged daily data by Fear & Greed classification and compare average trade PnL, win rate, and trade size.


In [1]:
order = ['Extreme Fear', 'Fear', 'Neutral', 'Greed', 'Extreme Greed']

summary = merged.groupby('classification').agg(
    avg_daily_pnl=('total_pnl', 'mean'),
    avg_trade_pnl=('avg_trade_pnl', 'mean'),
    avg_win_rate=('win_rate', 'mean'),
    avg_trade_size=('avg_size_usd', 'mean'),
    n_days=('date_only', 'count')
).reindex(order).round(2)

summary


                avg_daily_pnl  avg_trade_pnl  avg_win_rate  avg_trade_size  n_days
classification
Extreme Fear         52793.59          38.43         32.73         4091.80      14
Fear                 36891.82          31.28         32.91         6524.29      91
Neutral              19297.32          63.82         33.19         7157.53      67
Greed                11140.57          39.41         33.60         6735.30     193
Extreme Greed        23817.29          56.74         46.74         4410.52     114

## 6. Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

summary['avg_trade_pnl'].plot(kind='bar', ax=axes[0], color='#2b6cb0')
axes[0].set_title('Average PnL per Trade by Sentiment')
axes[0].set_ylabel('Avg Closed PnL ($)')
axes[0].set_xlabel('')

summary['avg_win_rate'].plot(kind='bar', ax=axes[1], color='#38a169')
axes[1].set_title('Average Win Rate by Sentiment')
axes[1].set_ylabel('Win Rate (%)')
axes[1].set_xlabel('')

plt.tight_layout()
plt.savefig('pnl_winrate_by_sentiment.png', dpi=150)
plt.show()


In [ ]:
merged['days_count'] = merged.groupby('classification')['classification'].transform('count')

fig, ax = plt.subplots(figsize=(7, 5))
merged['classification'].value_counts().reindex(order).plot(kind='bar', ax=ax, color='#805ad5')
ax.set_title('Number of Trading Days by Market Sentiment')
ax.set_ylabel('Days')
plt.tight_layout()
plt.savefig('days_by_sentiment.png', dpi=150)
plt.show()


## 7. Correlation Check

Does the raw Fear & Greed index value correlate directly with average trade PnL?


In [1]:
corr = merged['value'].corr(merged['avg_trade_pnl'])
print(f"Correlation between sentiment score and avg trade PnL: {corr:.3f}")


Correlation between sentiment score and avg trade PnL: 0.037


The near-zero correlation confirms that sentiment alone is a weak *linear* predictor of trade-level PnL. The signal shows up instead when sentiment is treated as a **categorical regime** (Fear vs. Greed vs. Neutral), where win rate and average trade size shift meaningfully between regimes even though the point-in-time index value doesn't.


## 8. Key Insights

1. **Win rate rises sharply in Extreme Greed** (~46.7%) versus all other regimes (~33%), suggesting momentum/trend-following conditions are easier to trade profitably.
2. **Average trade size is smallest during Extreme Fear and Extreme Greed** — traders size down at emotional extremes, likely as a risk-management response to volatility.
3. **Neutral markets produce the highest average PnL per trade ($63.82)** despite having no strong directional bias, hinting that traders may be more selective/disciplined when there's no dominant crowd emotion pushing them.
4. **Extreme Fear days are rare (14 of 479) but carry the highest total daily PnL on average** — likely dominated by a small number of large contrarian or liquidation-driven trades rather than broad-based profitability.
5. **The raw sentiment score has almost no linear correlation with PnL (r ≈ 0.04)** — sentiment should be used as a regime filter/context signal, not a direct predictive feature.

## 9. Suggested Trading Strategy Implications

- Treat Fear & Greed classification as a **regime filter** layered on top of an existing strategy, rather than a standalone signal.
- Consider **increasing position size selectively during Extreme Greed**, where win rate is structurally higher, while keeping standard risk controls.
- Use **Neutral regimes** for higher-conviction, lower-frequency setups given the higher average PnL per trade.
- Avoid oversizing during Extreme Fear/Extreme Greed transitions, consistent with observed trader behavior of derisking at extremes.
